# PARTIE 2 -- Optimisation et bilan d'architecture (après-midi)

## 2.1 Anatomie d'un plan d'exécution Spark

Quand vous soumettez une requête DataFrame ou SQL, Spark ne l'exécute pas
immédiatement. Il la fait d'abord passer par quatre phases de planification.

```
Code Python / SQL
       │
       ▼
  Plan non résolu (Unresolved Logical Plan)
  "Résolution" : vérification des noms de colonnes et des types
       │
       ▼
  Plan logique résolu (Resolved Logical Plan)
  "Analyse" : règles d'optimisation algébrique (push-down, élimination)
       │
       ▼
  Plan logique optimisé (Optimized Logical Plan)
  Catalyst applique ~70 règles de réécriture
       │
       ▼
  Plan(s) physique(s) (Physical Plans)
  Plusieurs stratégies sont énumérées (SortMerge, Broadcast, Hash...)
       │
       ▼
  Plan physique sélectionné (Selected Physical Plan)
  Tungsten génère le bytecode JVM optimisé (whole-stage codegen)
       │
       ▼
  Exécution distribuée
```

`explain(mode="formatted")` vous montre ces plans à la demande.


In [ ]:
# Illustration : requête simple -- que fait Catalyst ?
df_exemple = (
    df
    .filter(col("annee") == 2023)
    .filter(col("heure").between(7, 9))
    .groupBy("station_id")
    .agg(spark_avg("taux_occupation").alias("taux_pointe"))
    .filter(col("taux_pointe") < 0.2)   # filtre APRES agrégation
)

print("=== Plan logique (ce que vous avez écrit) ===")
df_exemple.explain(mode="simple")


In [ ]:
print("=== Plan physique optimisé (ce que Spark exécutera réellement) ===")
df_exemple.explain(mode="formatted")
# Observez :
# 1. Le filtre "annee = 2023" est pushé AVANT la lecture (PartitionFilter)
# 2. Le filtre "heure BETWEEN 7 AND 9" est appliqué pendant le scan (DataFilter)
# 3. L'agrégation utilise HashAggregate (plus rapide que SortAggregate)
# 4. Le filtre final "taux_pointe < 0.2" est appliqué APRÈS l'agrégation


---
## 2.2 Identifier et corriger un data skew

Le **data skew** (déséquilibre de données) est l'une des causes les plus
fréquentes de lenteur dans Spark. Il se produit quand les données ne sont
pas distribuées équitablement entre les partitions après un shuffle.

**Symptôme dans le Spark UI** : un stage dont la durée totale est dominée
par 1 ou 2 tâches alors que les autres finissent en quelques secondes.

### Simulation d'un skew


In [ ]:
# Simulation d'un DataFrame avec skew artificiel
# Une station (la station 0) représente 90% des données
import random
random.seed(SEED)

n_normal = 100_000
n_skewed  = 900_000

df_normal  = spark.range(n_normal).withColumn(
    "station_id", (F.rand(seed=SEED) * 100).cast("int") + 1
)
df_dominant = spark.range(n_skewed).withColumn(
    "station_id", F.lit(0)
)
df_skewed = df_normal.union(df_dominant).withColumn(
    "valeur", F.rand(seed=SEED)
)

print(f"DataFrame skewé : {df_skewed.count():,} lignes")
print("Distribution des 5 stations les plus fréquentes :")
df_skewed.groupBy("station_id").count().orderBy(F.desc("count")).show(5)


In [ ]:
# ── Approche naïve : groupBy direct ──────────────────────────────────────────
t0 = time.perf_counter()
df_skewed.groupBy("station_id").agg(spark_avg("valeur")).count()
t_naif = time.perf_counter() - t0
print(f"GroupBy naïf            : {t_naif:.2f} s")
print("  -> Observez dans le Spark UI : une tâche prend beaucoup plus longtemps")
print("     que les autres dans le stage du shuffle.")


In [ ]:
# ── Technique 1 : salting (ajout d'un sel aléatoire) ─────────────────────────
# On divise la clé dominante en N sous-groupes, on agrège partiellement,
# puis on supprime le sel et on agrège globalement.
N_SEL = 10

df_sale = df_skewed.withColumn(
    "cle_salee",
    F.concat(
        col("station_id").cast("string"),
        F.lit("_"),
        (F.rand(seed=SEED) * N_SEL).cast("int").cast("string")
    )
)

t0 = time.perf_counter()
# Agrégation partielle sur la clé salée
df_partiel = (
    df_sale
    .groupBy("cle_salee", "station_id")
    .agg(
        spark_avg("valeur").alias("moy_partielle"),
        F.count("*").alias("n_partielle")
    )
)
# Agrégation finale sur la clé originale (moyenne pondérée)
df_final_sale = (
    df_partiel
    .groupBy("station_id")
    .agg(
        (F.sum(col("moy_partielle") * col("n_partielle")) / F.sum("n_partielle"))
        .alias("moy_finale")
    )
)
df_final_sale.count()
t_sale = time.perf_counter() - t0

print(f"GroupBy avec salting     : {t_sale:.2f} s")
print(f"Gain                     : x{t_naif / t_sale:.1f}")


In [ ]:
# ── Technique 2 : broadcast de la petite table ────────────────────────────────
# Si l'une des tables d'une jointure est petite, on la broadcast
# pour éviter entièrement le shuffle de la grande table.

import time

df_grande = df.filter(col("annee") == 2023)   # ~6M lignes
df_petite = spark.createDataFrame(
    [(i, f"Catégorie {i % 4}") for i in range(1500)],
    ["station_id", "categorie"]
)

print(f"Grande table : {df_grande.count():,} lignes")
print(f"Petite table : {df_petite.count()} lignes")

# Sans broadcast : SortMergeJoin (shuffle des deux tables)
t0 = time.perf_counter()
n1 = df_grande.join(df_petite, on="station_id", how="inner").count()
t_smj = time.perf_counter() - t0

# Avec broadcast : BroadcastHashJoin (pas de shuffle de la grande table)
t0 = time.perf_counter()
n2 = df_grande.join(F.broadcast(df_petite), on="station_id", how="inner").count()
t_bcast = time.perf_counter() - t0

print(f"\nSortMergeJoin    : {t_smj:.2f} s  ({n1:,} lignes)")
print(f"BroadcastHashJoin: {t_bcast:.2f} s  ({n2:,} lignes)")
print(f"Gain             : x{t_smj / t_bcast:.1f}")
print()
print("Seuil automatique de broadcast (configurable) :")
print(f"  spark.sql.autoBroadcastJoinThreshold = "
      f"{spark.conf.get('spark.sql.autoBroadcastJoinThreshold')} bytes")


---
## 2.3 Partitionnement optimal

Le nombre de partitions a un impact direct sur les performances.
Trop peu : les tâches sont longues, les coeurs restent inactifs.
Trop de partitions : l'overhead de scheduling dépasse le gain du parallélisme.

**Règle empirique** : viser des partitions de 100 à 300 MB après décompression.
En mode local, aligner sur le nombre de coeurs disponibles.


In [ ]:
import os

n_coeurs = os.cpu_count()
print(f"Coeurs disponibles : {n_coeurs}")
print(f"spark.sql.shuffle.partitions (actuel) : "
      f"{spark.conf.get('spark.sql.shuffle.partitions')}")

# ── Impact du nombre de partitions sur un calcul itératif ──────────────────
df_test_part = df.filter(col("annee") == 2023)
n_lignes     = df_test_part.count()
print(f"\nDataFrame de test : {n_lignes:,} lignes")

for n_parts in [2, 4, 8, 16, 32]:
    spark.conf.set("spark.sql.shuffle.partitions", n_parts)
    t0 = time.perf_counter()
    (
        df_test_part
        .groupBy("station_id", "heure")
        .agg(spark_avg("taux_occupation"))
        .count()
    )
    t = time.perf_counter() - t0
    taille_part = n_lignes / n_parts
    print(f"  {n_parts:>3} partitions  ({taille_part:>8,.0f} lignes/partition) : {t:.2f} s")

# Restauration
spark.conf.set("spark.sql.shuffle.partitions", SHUFFLE_PARTS)


In [ ]:
# ── Repartitionnement explicite vs coalesce ───────────────────────────────────
# repartition(n) : shuffle complet, redistribution équilibrée
# coalesce(n)    : fusion de partitions adjacentes, sans shuffle
#                  (uniquement pour RÉDUIRE le nombre de partitions)

print("Partitions actuelles du DataFrame :", df.rdd.getNumPartitions())

t0 = time.perf_counter()
df_reparti = df.repartition(n_coeurs)
df_reparti.count()
t_reparti = time.perf_counter() - t0

t0 = time.perf_counter()
df_coalesce = df.coalesce(n_coeurs)
df_coalesce.count()
t_coalesce = time.perf_counter() - t0

print(f"repartition({n_coeurs}) : {t_reparti:.2f} s -- {df_reparti.rdd.getNumPartitions()} partitions")
print(f"coalesce({n_coeurs})    : {t_coalesce:.2f} s -- {df_coalesce.rdd.getNumPartitions()} partitions")
print()
print("Règle :")
print("  Augmenter ou équilibrer les partitions -> repartition() (shuffle)")
print("  Réduire les partitions avant une écriture -> coalesce() (pas de shuffle)")


---
## 2.4 Lecture avancée du Spark UI

Le Spark UI est votre principal outil de diagnostic. Voici les indicateurs
à surveiller dans chaque onglet.

### Onglet Jobs
- **Duration** : durée totale. Si un job est anormalement long, aller dans Stages.
- **Stages Skipped** : stages dont le résultat était en cache -- c'est bon signe.

### Onglet Stages
- **Task Distribution** : si la barre des durées est très étalée, il y a du skew.
- **Input / Shuffle Read / Shuffle Write** : un Shuffle Write élevé indique
  un shufflecoûteux. Chercher à le réduire par repartitionnement ou broadcast.
- **Spill (Memory / Disk)** : si non nul, Spark a dû écrire sur disque faute
  de mémoire. Augmenter `spark.driver.memory` ou réduire la taille des partitions.

### Onglet Storage
- Les DataFrames en cache apparaissent ici avec leur taille en mémoire et sur disque.
- Si **Fraction Cached < 1.0**, le cache a débordé. Utiliser `MEMORY_AND_DISK`.

### Onglet SQL
- Chaque requête DataFrame ou SQL génère un plan visualisé en DAG.
- Les noeuds en **orange** sont les shuffles (exchange operators).
- Les **métriques de chaque noeud** (lignes lues, lignes émises) permettent
  d'identifier les filtres peu sélectifs ou les jointures cartésiennes accidentelles.


In [ ]:
# Génération d'un job intentionnellement lent pour analyse dans le Spark UI
print("Génération d'un job avec shuffle visible dans le Spark UI...")
print("Ouvrez http://localhost:4040 -> SQL/DataFrame -> dernière requête")

df_analyse = (
    df
    .groupBy("station_id", "annee", "mois", "heure")
    .agg(
        spark_avg("taux_occupation").alias("taux_moy"),
        F.stddev("taux_occupation").alias("taux_std"),
        F.count("*").alias("n")
    )
    .filter(col("n") >= 10)
    .join(
        df.groupBy("station_id")
          .agg(spark_avg("taux_occupation").alias("taux_global")),
        on="station_id"
    )
    .withColumn("ecart_global", col("taux_moy") - col("taux_global"))
    .orderBy(F.desc("ecart_global"))
)

t0 = time.perf_counter()
n = df_analyse.count()
print(f"Résultat : {n:,} lignes en {time.perf_counter()-t0:.2f} s")
print("\nAllez maintenant dans Spark UI -> SQL/DataFrame -> dernière entrée.")
print("Identifiez : le nombre d'Exchange (shuffle), l'opération la plus coûteuse.")


---
## 2.5 Quand utiliser Spark -- et quand ne pas l'utiliser ?

Spark n'est pas la bonne réponse à tous les problèmes. Voici un guide de décision
construit à partir des expériences de ces trois jours.

| Critère | Pandas | PySpark |
|---------|--------|---------|
| Volume | < 10 GB en RAM | > 10 GB ou hors RAM |
| Latence requise | < 1 s (interactif) | secondes à minutes (batch) |
| Infrastructure | laptop / serveur seul | cluster ou machine puissante |
| Complexité d'installation | `pip install pandas` | JVM + config |
| Débogage | simple (Python pur) | plus complexe (Spark UI) |
| Streaming | non (ou limité) | oui (Structured Streaming) |
| ML distribué | scikit-learn | MLlib |
| SQL analytique | limité | natif et optimisé |

**Règle pratique** : commencez par Pandas. Si vous rencontrez des problèmes
de mémoire, de temps de calcul, ou si vous avez besoin de streaming ou de
distribution, migrez vers Spark -- généralement en changeant `pd.read_*`
en `spark.read.*` et en adaptant les agrégations.


In [ ]:
# Mesure finale comparative : même calcul en Pandas vs Spark
# (sur un sous-ensemble pour que Pandas puisse le tenir en mémoire)

df_sample_pd = df.filter(col("annee") == 2023).sample(0.1, seed=SEED).toPandas()
print(f"Sous-échantillon Pandas : {len(df_sample_pd):,} lignes")

# ── Pandas ────────────────────────────────────────────────────────────────────
import pandas as pd

t0 = time.perf_counter()
result_pd = (
    df_sample_pd
    .groupby(["station_id", "heure"])["taux_occupation"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .rename(columns={"mean": "taux_moy", "std": "taux_std", "count": "n"})
    .query("n >= 5")
    .sort_values("taux_moy", ascending=False)
)
t_pandas = time.perf_counter() - t0

# ── Spark ─────────────────────────────────────────────────────────────────────
df_sample_sp = df.filter(col("annee") == 2023).sample(0.1, seed=SEED)

t0 = time.perf_counter()
result_sp = (
    df_sample_sp
    .groupBy("station_id", "heure")
    .agg(
        spark_avg("taux_occupation").alias("taux_moy"),
        F.stddev("taux_occupation").alias("taux_std"),
        F.count("*").alias("n")
    )
    .filter(col("n") >= 5)
    .orderBy(F.desc("taux_moy"))
)
result_sp.count()
t_spark = time.perf_counter() - t0

print(f"\nPandas  : {t_pandas:.3f} s  ({len(result_pd):,} lignes résultat)")
print(f"Spark   : {t_spark:.2f} s  ({result_sp.count():,} lignes résultat)")
print()
print("Conclusion : sur ce volume, Pandas est plus rapide.")
print("Spark devient intéressant quand le volume dépasse la RAM disponible,")
print("ou quand on intègre batch + streaming + ML dans le même pipeline.")


---
## 2.6 Pipeline final bout-en-bout

Pour clore le cours, nous assemblons en un seul pipeline la chaîne complète :
ingestion Delta → feature engineering SQL → prédiction MLlib → résumé analytique.

C'est la démonstration qu'un pipeline Spark couvre l'intégralité du cycle
de vie de la donnée, du stockage brut à la valeur métier.


In [ ]:
from pyspark.ml import PipelineModel

print("=== Pipeline final ClimaCity Paris ===\n")

# ── 1. Ingestion depuis Delta ─────────────────────────────────────────────────
print("[1/5] Lecture depuis Delta Lake...")
t0 = time.perf_counter()
df_ingere = (
    spark.read.format("delta")
    .load(str(DELTA_DISPONIBLE))
    .filter(col("annee") == 2023)
)
print(f"      {df_ingere.count():,} snapshots chargés en {time.perf_counter()-t0:.1f}s")

# ── 2. Feature engineering (SQL) ──────────────────────────────────────────────
print("[2/5] Feature engineering...")
df_ingere.createOrReplaceTempView("ingere")

df_features_final = spark.sql("""
    SELECT *,
        SIN(heure    * 2 * 3.14159 / 24)  AS heure_sin,
        COS(heure    * 2 * 3.14159 / 24)  AS heure_cos,
        SIN(jour_sem * 2 * 3.14159 / 7)   AS joursem_sin,
        COS(jour_sem * 2 * 3.14159 / 7)   AS joursem_cos,
        SIN(mois     * 2 * 3.14159 / 12)  AS mois_sin,
        COS(mois     * 2 * 3.14159 / 12)  AS mois_cos,
        COALESCE(temperature_c, 13.0)     AS temperature_c_imp,
        COALESCE(humidite_pct, 75.0)      AS humidite_pct_imp,
        COALESCE(vent_kmh, 12.0)          AS vent_kmh_imp,
        COALESCE(precipitation_mm, 0.0)   AS precipitation_mm_imp,
        CAST(est_pluie   AS INT)           AS est_pluie_int,
        CAST(est_weekend AS INT)           AS est_weekend_int
    FROM ingere
    WHERE capacite > 0
""")

# Ajout des lags par fenêtre
fenetre = Window.partitionBy("station_id").orderBy("horodatage")
df_features_final = (
    df_features_final
    .withColumn("taux_lag1", F.lag("taux_occupation", 1).over(fenetre))
    .withColumn("taux_lag4", F.lag("taux_occupation", 4).over(fenetre))
    .withColumn("taux_moy4", spark_avg("taux_occupation").over(
        fenetre.rowsBetween(-4, -1)))
    .withColumn("cluster", F.lit(0))   # cluster par défaut (simplifié)
    .dropna(subset=FEATURES)
)
print(f"      {df_features_final.count():,} lignes avec features complètes")

# ── 3. Prédiction avec le modèle MLlib ────────────────────────────────────────
print("[3/5] Application du modèle GBT...")
model_prod = PipelineModel.load(str(chemin_model_local))
df_predictions = model_prod.transform(df_features_final)
print(f"      {df_predictions.count():,} prédictions générées")

# ── 4. Identification des stations à risque (taux prédit < 0.15) ──────────────
print("[4/5] Identification des stations à risque...")
df_risque = (
    df_predictions
    .filter(col("prediction") < 0.15)
    .groupBy("station_id", "nom_station", "code_arr", "heure")
    .agg(
        spark_round(spark_avg("prediction"), 3).alias("taux_predit_moy"),
        F.count("*").alias("nb_occurrences")
    )
    .orderBy("taux_predit_moy")
)
print(f"      {df_risque.count()} combinaisons station×heure à risque (taux prédit < 15%)")

# ── 5. Écriture du rapport en Delta ───────────────────────────────────────────
print("[5/5] Écriture du rapport...")
chemin_rapport = OUTPUT_DIR / "delta" / "rapport_risque"
(
    df_risque.write
    .format("delta")
    .mode("overwrite")
    .save(str(chemin_rapport))
)
print(f"      Rapport écrit dans {chemin_rapport}")

print("\n=== Pipeline terminé avec succès ===")


In [ ]:
# Lecture et affichage du rapport final
df_rapport = spark.read.format("delta").load(str(OUTPUT_DIR / "delta" / "rapport_risque"))

print("Top 20 des situations à risque (station × heure avec taux prédit le plus bas) :")
df_rapport.orderBy("taux_predit_moy").show(20, truncate=False)

print(f"\nRésumé par arrondissement :")
(
    df_rapport
    .groupBy("code_arr")
    .agg(
        F.count("*").alias("nb_situations_risque"),
        spark_round(spark_avg("taux_predit_moy"), 3).alias("taux_moyen_risque")
    )
    .orderBy(F.desc("nb_situations_risque"))
    .show(20)
)


---
## Bilan du Jour 3 et du projet

### Ce que nous avons fait

| Étape | Module | Concept clé |
|-------|--------|-------------|
| Features temporelles cycliques | MLlib / DataFrame | Encodage sin/cos, features de lag |
| Split temporel train/test | MLlib | Pas de fuite d'information |
| Clustering K-Means | MLlib | Pipeline, méthode du coude, `VectorAssembler` |
| Visualisation Folium | Python | Carte interactive des clusters |
| Régression GBT | MLlib | `GBTRegressor`, importance des features |
| Évaluation | MLlib | RMSE, MAE, R², `RegressionEvaluator` |
| Validation croisée | MLlib | `CrossValidator`, `ParamGridBuilder` |
| Tracking d'expériences | MLflow | `log_params`, `log_metrics`, `log_model` |
| Rechargement de modèle | MLflow / Spark | `mlflow.spark.load_model`, `PipelineModel.load` |
| Analyse du Catalyst | Spark | `explain(mode="formatted")`, plans logique/physique |
| Data skew | Spark | Salting, diagnostic Spark UI |
| Broadcast join | Spark | `broadcast()`, seuil automatique |
| Partitionnement | Spark | `repartition` vs `coalesce`, `shuffle.partitions` |
| Pipeline bout-en-bout | Spark | Delta → SQL → MLlib → Delta |

### Ce que vous savez faire après ces trois jours

À l'issue de ce projet, vous maîtrisez l'ensemble du spectre Spark :

- **Ingestion** : RDD, DataFrame, lecture Parquet/Delta avec predicate pushdown.
- **Transformation** : API DataFrame, Spark SQL, fenêtrage analytique, jointures.
- **Persistance** : Delta Lake ACID, time-travel, MERGE INTO.
- **Streaming** : source fichier, fenêtres glissantes, watermark, foreachBatch.
- **Machine Learning** : Pipeline MLlib, clustering, régression, validation croisée.
- **Observabilité** : Spark UI, MLflow, `explain()`.
- **Optimisation** : cache, broadcast, repartitionnement, diagnostic du skew.

### Prochaines étapes

Ce projet pose les bases. Pour aller plus loin :

- **Kafka** : remplacer la file source par un vrai broker pour le streaming.
- **Kubernetes / Databricks** : déployer sur un vrai cluster.
- **MLflow Model Registry** : versionner et promouvoir les modèles en production.
- **Delta Live Tables** : orchestration déclarative de pipelines Delta.
- **GraphFrames** : analyser le réseau de stations comme un graphe.


In [ ]:
spark.stop()
print("SparkSession arrêtée. Projet ClimaCity Paris terminé !")
